# Print single tag yield table
## Make a nice table for the paper and MEMO

### Load utility functions

In [1]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Function for parsing the single tag yields

In [2]:
std::pair<double, double> GetSTYield(const std::string &Tag) {
    std::pair<double, double> Yield{0.0, 0.0};
    std::string Filename, VariableName;
    Filename = "/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/SingleTagFit/";
    Filename += Tag + "/" + Tag + "_MBC_FitResults.txt";
    VariableName = Tag + "_SingleTag_Yield";
    std::ifstream Infile(Filename);
    std::string Line;
    while(std::getline(Infile, Line)) {
        std::stringstream ss(Line);
        std::string Name;
        double Value;
        ss >> Name >> Value;
        if(Name == VariableName) {
            Yield.first = Value;
        } else if(Name == VariableName + "_err") {
            Yield.second = Value;
        }
    }
    return Yield;
}

### Function for parsing the single tag efficiencies

In [3]:
std::pair<double, double> GetSTEff(const std::string &Tag) {
    std::map<std::string, double> Efficiencies;
    std::string Filename("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/CommonInputs/");
    Filename += "Efficiencies/SingleTagEfficiencies.txt";
    std::ifstream File(Filename);
    std::string Line;
    while(std::getline(File, Line)) {
        if(Line.empty()) {
            continue;
        }
        std::string Name;
        double Value;
        std::stringstream ss(Line);
        ss >> Name >> Value;
        Efficiencies.insert({Name, Value});
    }
    File.close();
    std::string Label = Tag + "_SingleTagEfficiency";
    return {Efficiencies[Label], Efficiencies[Label + "_err"]};
}

### Function for rounding yields to three significant figures

In [4]:
std::pair<double, double> Round3SigFigs(const std::pair<double, double> &Value) {
    double Factor = std::pow(10.0, 3 - std::ceil(std::log10(std::fabs(Value.second))));
    std::pair<double, double> RoundedValue;
    RoundedValue.first = std::round(Value.first*Factor)/Factor;
    RoundedValue.second = std::round(Value.second*Factor)/Factor;
    return RoundedValue;
}

### List of tag modes

In [5]:
std::vector<std::string> TagModesCP{"KK",
                                    "pipi",
                                    "pipipi0",
                                    "KSpi0pi0",
                                    "KLpi0",
                                    "KSpi0",
                                    "KSeta",
                                    "KSetaPrimepipieta",
                                    "KSetaPrimerhogamma",
                                    "KSpipipi0"};
std::vector<std::string> TagModesFlavour{"Kpi", "Kpipi0", "Kpipipi", "KeNu"};
std::vector<std::string> TagModesSCMB{"KSpipi", "KLpipi"};
std::vector<std::string> TagModes{"KKpipi"};
TagModes.insert(TagModes.end(), TagModesFlavour.begin(), TagModesFlavour.end());
TagModes.insert(TagModes.end(), TagModesCP.begin(), TagModesCP.end());
TagModes.insert(TagModes.end(), TagModesSCMB.begin(), TagModesSCMB.end());

In [6]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 20;
std::cout << Blank << "\\toprule" << "\n";
std::cout << Blank;
std::cout << std::left << std::setw(33) << "Tag mode" << " & "
          << std::left << std::setw(Width) << "Single tag yield" << " & "
          << std::left << std::setw(Width) << "Single tag efficiency ($\\%$)" << " \\\\" << "\n";
std::cout << Blank << "\\midrule" << "\n";
for(const auto &Tag : TagModes) {
    if(Tag == "Kpi" || Tag == "KK" || Tag == "KSpipi") {
        std::cout << Blank << "\\midrule" << "\n";
    }
    std::cout << Blank;
    std::cout << std::left << std::setw(33);
    std::cout << GetTagName(Tag) << " & ";
    std::cout << std::fixed << std::setprecision(0);
    std::cout << std::left << std::setw(Width);
    auto Yield = GetSTYield(Tag);
    Yield = Round3SigFigs(Yield);
    std::cout << PrintLaTeXNumber(Yield.first, Yield.second) << " & ";
    std::cout << std::left << std::setw(Width);
    auto Efficiency = GetSTEff(Tag);
    std::cout << PrintLaTeXNumber(Efficiency.first*100.0, Efficiency.second*100.0) << " \\\\" << "\n";
}
std::cout << Blank << "\\bottomrule" << "\n";

        \toprule
        Tag mode                          & Single tag yield     & Single tag efficiency ($\%$) \\
        \midrule
        $\kaonp\kaonm\pip\pim$            & $74487 \pm 426$      & $18.71 \pm 0.04$     \\
        \midrule
        $\kaonm\pip$                      & $3817700 \pm 2040$   & $68.53 \pm 0.05$     \\
        $\kaonm\pip\piz$                  & $7252630 \pm 9050$   & $36.90 \pm 0.05$     \\
        $\kaonm\pip\pim\pip$              & $4847930 \pm 2710$   & $41.14 \pm 0.05$     \\
        $\kaonm e^{+}\nu_e$               & $3030700 \pm 41200$  & $58.26 \pm 0.17$     \\
        \midrule
        $\kaonp\kaonm$                    & $387176 \pm 677$     & $63.10 \pm 0.05$     \\
        $\pip\pim$                        & $142731 \pm 475$     & $66.65 \pm 0.05$     \\
        $\pip\pim\piz$                    & $757190 \pm 1500$    & $36.27 \pm 0.05$     \\
        $\kshort\piz\piz$                 & $159212 \pm 616$     & $14.46 \pm 0.04$     \\
        $\klon